# Week 4 — SQL & Data Wrangling

## Day 1: SELECT, WHERE, ORDER BY, LIMIT

ยินดีต้อนรับสู่สัปดาห์ที่ 4 ครับ สัปดาห์นี้เราจะเปลี่ยนจาก pandas (Python) มาเป็น **SQL**

### Day 1 จะได้อะไรบ้าง

1. รู้จัก SQL คืออะไรและทำไมต้องเรียน
2. เชื่อมต่อกับ SQLite database ผ่าน Python
3. เขียน query พื้นฐาน: `SELECT`, `WHERE`, `ORDER BY`, `LIMIT`, `DISTINCT`
4. ลองทำแบบฝึกหัด Day 1

เวลา: ~2 ชั่วโมง

## 1. SQL คืออะไร

**SQL** (Structured Query Language) คือภาษาที่ใช้ "คุย" กับฐานข้อมูล (database)

- **Excel** = ดูข้อมูลทีละ sheet ด้วยตา
- **pandas** = เขียน Python จัดการ DataFrame
- **SQL** = บอกฐานข้อมูลว่า "อยากได้อะไร" แล้วระบบจัดการให้

> วันนี้เราใช้ **SQLite** ซึ่งเป็นฐานข้อมูลแบบไฟล์เดียว (ไฟล์ `ecommerce.db`) ไม่ต้องลงเซิร์ฟเวอร์๊

## 2. Setup

เราจะใช้ library 2 ตัว:

- `sqlite3` — มากับ Python อยู่แล้ว ใช้คุยกับไฟล์ `.db`
- `pandas` — เพื่อแสดงผล query เป็นตารางสวย ๆ

In [20]:
import sqlite3
import pandas as pd
from pathlib import Path
import random
from datetime import datetime, timedelta

DB_PATH = Path(r'C:\Users\Rutanech\OneDrive - Thammasat University\project\Data Scientist Boostcamp\week04_sql_data_wrangling\ecommerce.db')
conn = sqlite3.connect(str(DB_PATH))

def q(sql):
    return pd.read_sql_query(sql, conn)

# สร้าง database อัตโนมัติถ้ายังไม่มีข้อมูล
tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
if 'customers' not in tables:
    print('กำลังสร้าง database...')
    cur = conn.cursor()
    cur.executescript("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
            email TEXT, city TEXT, country TEXT, signup_date TEXT);
        CREATE TABLE IF NOT EXISTS products (
            product_id INTEGER PRIMARY KEY, product_name TEXT,
            category TEXT, price REAL, cost REAL);
        CREATE TABLE IF NOT EXISTS orders (
            order_id INTEGER PRIMARY KEY, customer_id INTEGER,
            order_date TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS order_items (
            order_item_id INTEGER PRIMARY KEY AUTOINCREMENT, order_id INTEGER,
            product_id INTEGER, quantity INTEGER, unit_price REAL);
    """)
    random.seed(42)
    first_names = ['Somchai','Suda','Anan','Nicha','Krit','Ploy','Tul','Mint','Boss','Fern',
                   'John','Emily','Hiro','Yui','Mei','Wei','Ravi','Aisha','Lars','Marco',
                   'Sara','Daan','Kai','Niran','Praew','Ton','Beam','Aom','Mook','Jane']
    last_names  = ['Wong','Suk','Chai','Ito','Smith','Garcia','Singh','Kim','Tan','Park',
                   'Phan','Lee','Brown','Anand','Lim','Sato','Wang','Patel','Mueller','Costa']
    cities = [('Bangkok','Thailand'),('Chiang Mai','Thailand'),('Phuket','Thailand'),
              ('Singapore','Singapore'),('Tokyo','Japan'),('Osaka','Japan'),
              ('Seoul','South Korea'),('Hanoi','Vietnam'),('Jakarta','Indonesia'),
              ('Manila','Philippines'),('Kuala Lumpur','Malaysia'),('Taipei','Taiwan')]
    customers_data = []
    start = datetime(2024, 1, 1)
    for i in range(1, 51):
        fn = random.choice(first_names); ln = random.choice(last_names)
        city, country = random.choice(cities)
        signup = start + timedelta(days=random.randint(0, 500))
        customers_data.append((i, fn, ln, f'{fn.lower()}.{ln.lower()}{i}@example.com',
                               city, country, signup.strftime('%Y-%m-%d')))
    cur.executemany('INSERT INTO customers VALUES (?,?,?,?,?,?,?)', customers_data)
    products_data = [
        ('Wireless Mouse','Electronics',590,250),('Mechanical Keyboard','Electronics',2490,1200),
        ('USB-C Cable','Electronics',190,60),('Bluetooth Speaker','Electronics',1290,600),
        ('Laptop Stand','Accessories',790,300),('Notebook A5','Stationery',120,40),
        ('Ballpoint Pen Pack','Stationery',90,25),('Sticky Notes','Stationery',60,15),
        ('Coffee Mug','Home',250,80),('Desk Lamp','Home',990,400),
        ('Yoga Mat','Sports',690,250),('Water Bottle','Sports',290,100),
        ('Running Shoes','Sports',2890,1300),('Backpack','Accessories',1490,600),
        ('Sunglasses','Accessories',890,350),('T-Shirt Plain','Apparel',390,150),
        ('Cap','Apparel',290,100),('Hoodie','Apparel',990,420),
        ('Smart Watch','Electronics',4990,2300),('Headphones','Electronics',1990,900),
    ]
    for pid, (name, cat, price, cost) in enumerate(products_data, start=1):
        cur.execute('INSERT INTO products VALUES (?,?,?,?,?)', (pid, name, cat, price, cost))
    statuses = ['completed','completed','completed','completed','cancelled','refunded','pending']
    oid = 1; order_rows = []; item_rows = []
    for cust_id, *_, signup_str in customers_data:
        signup_dt = datetime.strptime(signup_str, '%Y-%m-%d')
        for _ in range(random.choices([0,1,2,3,4,5,6,7], weights=[2,3,4,4,3,2,1,1])[0]):
            order_dt = signup_dt + timedelta(days=random.randint(1, 540))
            if order_dt > datetime(2026, 5, 16): continue
            order_rows.append((oid, cust_id, order_dt.strftime('%Y-%m-%d'), random.choice(statuses)))
            for pid in random.sample(range(1, 21), random.randint(1, 4)):
                up = products_data[pid-1][2]
                if random.random() < 0.15: up = round(up * 0.9, 2)
                item_rows.append((oid, pid, random.randint(1, 3), up))
            oid += 1
    cur.executemany('INSERT INTO orders VALUES (?,?,?,?)', order_rows)
    cur.executemany('INSERT INTO order_items (order_id,product_id,quantity,unit_price) VALUES (?,?,?,?)', item_rows)
    conn.commit()
    print('สร้าง database เสร็จแล้ว (customers=50, products=20, orders=', len(order_rows), ')')

print('connected to ecommerce.db')
q("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")

กำลังสร้าง database...
สร้าง database เสร็จแล้ว (customers=50, products=20, orders= 128 )
connected to ecommerce.db


,name
0,customers
1,products
2,orders
3,order_items


## 3. สำรวจ schema ของ database

ก่อนเขียน query สิ่งแรกที่ต้องทำในงานจริงคือ **"รู้ก่อนว่ามีตารางอะไรบ้าง และแต่ละตารางมี column อะไร"**

In [21]:
# ดูว่ามีตารางอะไรบ้าง
q("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
""")

,name
0,customers
1,products
2,orders
3,order_items


In [22]:
# ดู column ของตาราง customers
q("PRAGMA table_info(customers)")

,cid,name,type,notnull,dflt_value,pk
0,0,customer_id,INTEGER,0,None,1
1,1,first_name,TEXT,0,None,0
2,2,last_name,TEXT,0,None,0
3,3,email,TEXT,0,None,0
4,4,city,TEXT,0,None,0
5,5,country,TEXT,0,None,0
6,6,signup_date,TEXT,0,None,0


### Schema สรุป

ใน DB ของเรามี 4 ตาราง:

- `customers` — ข้อมูลลูกค้า (customer_id, first_name, last_name, email, city, country, signup_date)
- `products` — สินค้า (product_id, product_name, category, price, cost)
- `orders` — ออเดอร์ (order_id, customer_id, order_date, status)
- `order_items` — รายการสินค้าในแต่ละออเดอร์ (order_item_id, order_id, product_id, quantity, unit_price)

## 4. ดูข้อมูลตัวอย่าง

**`SELECT *`** = เลือกทุก column (`*` แปลว่า "ทั้งหมด")

In [23]:
q("SELECT * FROM customers LIMIT 5")

,customer_id,first_name,last_name,email,city,country,signup_date
0,1,Sara,Ito,sara.ito1@example.com,Bangkok,Thailand,2025-01-14
1,2,Boss,Kim,boss.kim2@example.com,Singapore,Singapore,2024-03-12
2,3,Niran,Ito,niran.ito3@example.com,Kuala Lumpur,Malaysia,2025-01-14
3,4,Mook,Patel,mook.patel4@example.com,Chiang Mai,Thailand,2024-10-29
4,5,Yui,Suk,yui.suk5@example.com,Bangkok,Thailand,2024-02-17


In [24]:
q("SELECT * FROM products LIMIT 5")

,product_id,product_name,category,price,cost
0,1,Wireless Mouse,Electronics,590.0,250.0
1,2,Mechanical Keyboard,Electronics,2490.0,1200.0
2,3,USB-C Cable,Electronics,190.0,60.0
3,4,Bluetooth Speaker,Electronics,1290.0,600.0
4,5,Laptop Stand,Accessories,790.0,300.0


In [25]:
q("SELECT * FROM orders LIMIT 5")

,order_id,customer_id,order_date,status
0,1,1,2025-02-03,completed
1,2,1,2025-05-23,completed
2,3,2,2025-03-30,completed
3,4,2,2024-11-02,completed
4,5,3,2025-08-22,cancelled


In [26]:
q("SELECT * FROM order_items LIMIT 5")

,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,10,3,990.0
1,2,1,8,1,60.0
2,3,1,2,1,2490.0
3,4,2,18,2,990.0
4,5,2,6,3,120.0


## 5. SELECT — เลือก column ที่ต้องการ

```sql
SELECT column1, column2
FROM table_name;
```

In [27]:
q("""
SELECT first_name, last_name, country
FROM customers
LIMIT 10
""")

,first_name,last_name,country
0,Sara,Ito,Thailand
1,Boss,Kim,Singapore
2,Niran,Ito,Malaysia
3,Mook,Patel,Thailand
4,Yui,Suk,Thailand
5,Tul,Kim,Indonesia
6,Somchai,Patel,Singapore
7,Sara,Patel,South Korea
8,Mei,Mueller,Japan
9,Aom,Wong,Thailand


### ใช้ alias (`AS`) เปลี่ยนชื่อ column

In [28]:
q("""
SELECT
    first_name AS firstname,
    last_name  AS lastname,
    country    AS nationality
FROM customers
LIMIT 5
""")

,firstname,lastname,nationality
0,Sara,Ito,Thailand
1,Boss,Kim,Singapore
2,Niran,Ito,Malaysia
3,Mook,Patel,Thailand
4,Yui,Suk,Thailand


### คำนวณ column ใหม่ระหว่าง SELECT

In [29]:
q("""
SELECT
    product_name,
    price,
    cost,
    price - cost AS profit_per_unit
FROM products
LIMIT 10
""")

,product_name,price,cost,profit_per_unit
0,Wireless Mouse,590.0,250.0,340.0
1,Mechanical Keyboard,2490.0,1200.0,1290.0
2,USB-C Cable,190.0,60.0,130.0
3,Bluetooth Speaker,1290.0,600.0,690.0
4,Laptop Stand,790.0,300.0,490.0
5,Notebook A5,120.0,40.0,80.0
6,Ballpoint Pen Pack,90.0,25.0,65.0
7,Sticky Notes,60.0,15.0,45.0
8,Coffee Mug,250.0,80.0,170.0
9,Desk Lamp,990.0,400.0,590.0


## 6. WHERE — กรองแถวที่ต้องการ

| ตัวดำเนินการ | ใช้ทำอะไร |
|---|---|
| `=` | เท่ากับ |
| `!=` | ไม่เท่ากับ |
| `>` `<` `>=` `<=` | มากกว่า / น้อยกว่า |
| `AND` `OR` | รวมเงื่อนไข |
| `IN` | อยู่ในชุดค่า |
| `BETWEEN` | อยู่ในช่วง |
| `LIKE` | match pattern |
| `IS NULL` | ตรวจ null |

In [30]:
q("""
SELECT first_name, last_name, city, country
FROM customers
WHERE country = 'Thailand'
""")

,first_name,last_name,city,country
0,Sara,Ito,Bangkok,Thailand
1,Mook,Patel,Chiang Mai,Thailand
2,Yui,Suk,Bangkok,Thailand
3,Aom,Wong,Phuket,Thailand
4,Tul,Phan,Chiang Mai,Thailand
5,Jane,Brown,Chiang Mai,Thailand
6,Tul,Chai,Bangkok,Thailand
7,Mint,Park,Chiang Mai,Thailand
8,Mei,Lee,Phuket,Thailand
9,Krit,Tan,Phuket,Thailand


In [31]:
q("""
SELECT product_name, category, price
FROM products
WHERE price > 1000
""")

,product_name,category,price
0,Mechanical Keyboard,Electronics,2490.0
1,Bluetooth Speaker,Electronics,1290.0
2,Running Shoes,Sports,2890.0
3,Backpack,Accessories,1490.0
4,Smart Watch,Electronics,4990.0
5,Headphones,Electronics,1990.0


In [32]:
q("""
SELECT product_name, category, price
FROM products
WHERE category = 'Electronics'
  AND price < 2000
""")

,product_name,category,price
0,Wireless Mouse,Electronics,590.0
1,USB-C Cable,Electronics,190.0
2,Bluetooth Speaker,Electronics,1290.0
3,Headphones,Electronics,1990.0


In [33]:
q("""
SELECT first_name, last_name, city
FROM customers
WHERE city IN ('Bangkok', 'Chiang Mai', 'Phuket')
""")

,first_name,last_name,city
0,Sara,Ito,Bangkok
1,Mook,Patel,Chiang Mai
2,Yui,Suk,Bangkok
3,Aom,Wong,Phuket
4,Tul,Phan,Chiang Mai
5,Jane,Brown,Chiang Mai
6,Tul,Chai,Bangkok
7,Mint,Park,Chiang Mai
8,Mei,Lee,Phuket
9,Krit,Tan,Phuket


In [34]:
q("""
SELECT product_name, price
FROM products
WHERE price BETWEEN 500 AND 2000
""")

,product_name,price
0,Wireless Mouse,590.0
1,Bluetooth Speaker,1290.0
2,Laptop Stand,790.0
3,Desk Lamp,990.0
4,Yoga Mat,690.0
5,Backpack,1490.0
6,Sunglasses,890.0
7,Hoodie,990.0
8,Headphones,1990.0


In [35]:
q("""
SELECT product_name, category
FROM products
WHERE product_name LIKE '%Bottle%'
   OR product_name LIKE '%Cable%'
""")

,product_name,category
0,USB-C Cable,Electronics
1,Water Bottle,Sports


## 7. ORDER BY — เรียงผลลัพธ์

- `ASC` = น้อยไปมาก (default)
- `DESC` = มากไปน้อย

In [36]:
q("""
SELECT product_name, category, price
FROM products
ORDER BY price DESC
LIMIT 5
""")

,product_name,category,price
0,Smart Watch,Electronics,4990.0
1,Running Shoes,Sports,2890.0
2,Mechanical Keyboard,Electronics,2490.0
3,Headphones,Electronics,1990.0
4,Backpack,Accessories,1490.0


In [37]:
q("""
SELECT product_name, category, price
FROM products
ORDER BY category ASC, price DESC
LIMIT 15
""")

,product_name,category,price
0,Backpack,Accessories,1490.0
1,Sunglasses,Accessories,890.0
2,Laptop Stand,Accessories,790.0
3,Hoodie,Apparel,990.0
4,T-Shirt Plain,Apparel,390.0
5,Cap,Apparel,290.0
6,Smart Watch,Electronics,4990.0
7,Mechanical Keyboard,Electronics,2490.0
8,Headphones,Electronics,1990.0
9,Bluetooth Speaker,Electronics,1290.0


## 8. LIMIT — จำกัดจำนวนแถว

In [38]:
q("""
SELECT first_name, last_name, signup_date
FROM customers
ORDER BY signup_date DESC
LIMIT 3
""")

,first_name,last_name,signup_date
0,Praew,Garcia,2025-05-03
1,Niran,Mueller,2025-04-04
2,Anan,Singh,2025-03-24


## 9. DISTINCT — เอาเฉพาะค่าที่ไม่ซ้ำ

In [39]:
q("SELECT DISTINCT country FROM customers ORDER BY country")

,country
0,Indonesia
1,Japan
2,Malaysia
3,Philippines
4,Singapore
5,South Korea
6,Taiwan
7,Thailand
8,Vietnam


In [40]:
q("""
SELECT DISTINCT country, city
FROM customers
ORDER BY country, city
""")

,country,city
0,Indonesia,Jakarta
1,Japan,Osaka
2,Japan,Tokyo
3,Malaysia,Kuala Lumpur
4,Philippines,Manila
5,Singapore,Singapore
6,South Korea,Seoul
7,Taiwan,Taipei
8,Thailand,Bangkok
9,Thailand,Chiang Mai


## 10. ตัวอย่างเต็ม — ลำดับการเขียน SQL

```sql
SELECT   column1, column2
FROM     table
WHERE    <condition>
ORDER BY column [ASC|DESC]
LIMIT    n;
```

**ลำดับสำคัญ** — เขียนผิดลำดับจะ error

In [41]:
q("""
SELECT first_name, last_name, country, signup_date
FROM customers
WHERE country IN ('Thailand', 'Japan', 'Singapore', 'South Korea', 'Vietnam')
ORDER BY signup_date ASC
LIMIT 5
""")

,first_name,last_name,country,signup_date
0,Yui,Garcia,Vietnam,2024-01-02
1,Krit,Wang,Vietnam,2024-02-16
2,Yui,Suk,Thailand,2024-02-17
3,Tul,Phan,Thailand,2024-02-17
4,Boss,Kim,Singapore,2024-03-12


## 11. แบบฝึกหัด Day 1

ลองเขียน query แต่ละข้อด้วยตัวเองก่อน โจทย์ดูในไฟล์ `week04_exercises.md`

**Tip:** ถ้าติด ให้ดูตัวอย่างด้านบนเป็น reference ก่อน อย่ารีบดูเฉลย

### ข้อ 1: แสดง `customer_id`, `first_name`, `last_name`, `country` ของทุกคน

In [42]:
# เขียน query ของคุณตรงนี้
q("""
SELECT customer_id, first_name, last_name, country FROM customers 
""")

,customer_id,first_name,last_name,country
0,1,Sara,Ito,Thailand
1,2,Boss,Kim,Singapore
2,3,Niran,Ito,Malaysia
3,4,Mook,Patel,Thailand
4,5,Yui,Suk,Thailand
5,6,Tul,Kim,Indonesia
6,7,Somchai,Patel,Singapore
7,8,Sara,Patel,South Korea
8,9,Mei,Mueller,Japan
9,10,Aom,Wong,Thailand


### ข้อ 2: แสดงลูกค้าเฉพาะที่อยู่ประเทศ Thailand

In [44]:
# เขียน query ของคุณตรงนี้
q("""
SELECT * FROM customers WHERE country = "Thailand"
""")

,customer_id,first_name,last_name,email,city,country,signup_date
0,1,Sara,Ito,sara.ito1@example.com,Bangkok,Thailand,2025-01-14
1,4,Mook,Patel,mook.patel4@example.com,Chiang Mai,Thailand,2024-10-29
2,5,Yui,Suk,yui.suk5@example.com,Bangkok,Thailand,2024-02-17
3,10,Aom,Wong,aom.wong10@example.com,Phuket,Thailand,2024-12-23
4,12,Tul,Phan,tul.phan12@example.com,Chiang Mai,Thailand,2024-02-17
5,16,Jane,Brown,jane.brown16@example.com,Chiang Mai,Thailand,2024-10-09
6,18,Tul,Chai,tul.chai18@example.com,Bangkok,Thailand,2024-12-04
7,19,Mint,Park,mint.park19@example.com,Chiang Mai,Thailand,2025-03-13
8,21,Mei,Lee,mei.lee21@example.com,Phuket,Thailand,2024-07-08
9,32,Krit,Tan,krit.tan32@example.com,Phuket,Thailand,2024-05-06


### ข้อ 3: แสดงลูกค้าที่อยู่ Bangkok หรือ Chiang Mai (ใช้ `IN`)

In [45]:
# เขียน query ของคุณตรงนี้
q("""
SELECT * FROM customers WHERE city IN ("Bangkok", "Chiang Mai")
""")

,customer_id,first_name,last_name,email,city,country,signup_date
0,1,Sara,Ito,sara.ito1@example.com,Bangkok,Thailand,2025-01-14
1,4,Mook,Patel,mook.patel4@example.com,Chiang Mai,Thailand,2024-10-29
2,5,Yui,Suk,yui.suk5@example.com,Bangkok,Thailand,2024-02-17
3,12,Tul,Phan,tul.phan12@example.com,Chiang Mai,Thailand,2024-02-17
4,16,Jane,Brown,jane.brown16@example.com,Chiang Mai,Thailand,2024-10-09
5,18,Tul,Chai,tul.chai18@example.com,Bangkok,Thailand,2024-12-04
6,19,Mint,Park,mint.park19@example.com,Chiang Mai,Thailand,2025-03-13
7,37,Praew,Suk,praew.suk37@example.com,Chiang Mai,Thailand,2024-03-19
8,43,Sara,Phan,sara.phan43@example.com,Chiang Mai,Thailand,2024-05-30
9,46,Ploy,Wang,ploy.wang46@example.com,Chiang Mai,Thailand,2025-03-21


### ข้อ 4: แสดงรายชื่อ `country` ที่ไม่ซ้ำกัน

In [46]:
# เขียน query ของคุณตรงนี้
q("""
SELECT DISTINCT country FROM customers
""")

,country
0,Thailand
1,Singapore
2,Malaysia
3,Indonesia
4,South Korea
5,Japan
6,Philippines
7,Taiwan
8,Vietnam


### ข้อ 5: แสดงสินค้าทั้งหมด เรียงจากราคาแพงไปถูก

In [47]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM products
ORDER BY price DESC
""")

,product_id,product_name,category,price,cost
0,19,Smart Watch,Electronics,4990.0,2300.0
1,13,Running Shoes,Sports,2890.0,1300.0
2,2,Mechanical Keyboard,Electronics,2490.0,1200.0
3,20,Headphones,Electronics,1990.0,900.0
4,14,Backpack,Accessories,1490.0,600.0
5,4,Bluetooth Speaker,Electronics,1290.0,600.0
6,10,Desk Lamp,Home,990.0,400.0
7,18,Hoodie,Apparel,990.0,420.0
8,15,Sunglasses,Accessories,890.0,350.0
9,5,Laptop Stand,Accessories,790.0,300.0


### ข้อ 6: สินค้าที่ราคาอยู่ระหว่าง 500–2000 บาท (ใช้ `BETWEEN`)

In [48]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM products
WHERE price BETWEEN 500 AND 2000
""")

,product_id,product_name,category,price,cost
0,1,Wireless Mouse,Electronics,590.0,250.0
1,4,Bluetooth Speaker,Electronics,1290.0,600.0
2,5,Laptop Stand,Accessories,790.0,300.0
3,10,Desk Lamp,Home,990.0,400.0
4,11,Yoga Mat,Sports,690.0,250.0
5,14,Backpack,Accessories,1490.0,600.0
6,15,Sunglasses,Accessories,890.0,350.0
7,18,Hoodie,Apparel,990.0,420.0
8,20,Headphones,Electronics,1990.0,900.0


### ข้อ 7: สินค้า Electronics ที่ราคาน้อยกว่า 2000 บาท เรียงราคาจากน้อยไปมาก

In [60]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM products
WHERE price < 2000 AND category = "Electronics"
ORDER BY price ASC
""")

,product_id,product_name,category,price,cost
0,3,USB-C Cable,Electronics,190.0,60.0
1,1,Wireless Mouse,Electronics,590.0,250.0
2,4,Bluetooth Speaker,Electronics,1290.0,600.0
3,20,Headphones,Electronics,1990.0,900.0


### ข้อ 8: หาสินค้าที่ชื่อมีคำว่า "Pen" (ใช้ `LIKE`)

In [64]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM products
WHERE product_name LIKE "%Pen%"
""")

,product_id,product_name,category,price,cost
0,7,Ballpoint Pen Pack,Stationery,90.0,25.0


### ข้อ 9: ลูกค้าที่ลงทะเบียนหลังวันที่ 2024-06-30

In [70]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM customers
WHERE signup_date > "2024-06-30"
""")

,customer_id,first_name,last_name,email,city,country,signup_date
0,1,Sara,Ito,sara.ito1@example.com,Bangkok,Thailand,2025-01-14
1,3,Niran,Ito,niran.ito3@example.com,Kuala Lumpur,Malaysia,2025-01-14
2,4,Mook,Patel,mook.patel4@example.com,Chiang Mai,Thailand,2024-10-29
3,6,Tul,Kim,tul.kim6@example.com,Jakarta,Indonesia,2024-11-04
4,7,Somchai,Patel,somchai.patel7@example.com,Singapore,Singapore,2025-01-01
5,9,Mei,Mueller,mei.mueller9@example.com,Tokyo,Japan,2025-02-18
6,10,Aom,Wong,aom.wong10@example.com,Phuket,Thailand,2024-12-23
7,13,Hiro,Ito,hiro.ito13@example.com,Osaka,Japan,2025-03-09
8,14,Emily,Costa,emily.costa14@example.com,Tokyo,Japan,2025-02-17
9,16,Jane,Brown,jane.brown16@example.com,Chiang Mai,Thailand,2024-10-09


### ข้อ 10: Top 5 สินค้าที่กำไรต่อชิ้นสูงสุด (`price - cost`)

In [72]:
# เขียน query ของคุณตรงนี้
q("""
SELECT *
FROM products
ORDER BY price-cost DESC
LIMIT 5
""")

,product_id,product_name,category,price,cost
0,19,Smart Watch,Electronics,4990.0,2300.0
1,13,Running Shoes,Sports,2890.0,1300.0
2,2,Mechanical Keyboard,Electronics,2490.0,1200.0
3,20,Headphones,Electronics,1990.0,900.0
4,14,Backpack,Accessories,1490.0,600.0


## สรุป Day 1

วันนี้คุณได้เรียน:

- โครงสร้าง SQL พื้นฐาน `SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT`
- ตัวดำเนินการกรองข้อมูล: `=`, `!=`, `>`, `<`, `AND`, `OR`, `IN`, `BETWEEN`, `LIKE`
- `DISTINCT` หาค่าที่ไม่ซ้ำ
- เชื่อมต่อ SQLite ผ่าน Python และดูผลลัพธ์เป็น pandas DataFrame

### Checklist ก่อนข้ามไป Day 2

- [ ] ทำแบบฝึกหัด 10 ข้อด้านบนเสร็จ
- [ ] เขียน query ได้โดยไม่ดูเฉลยอย่างน้อย 7/10 ข้อ
- [ ] จดสิ่งที่เรียนรู้ใน `learning_log.md`

**อย่าลืม commit งานวันนี้ขึ้น GitHub ด้วยนะครับ**

In [ ]:
# ปิด connection เมื่อทำเสร็จ
conn.close()
print('connection closed')